# Inspect missing instance-label IDs

This notebook inspects the five instance labels rejected during chip-creation profiling. It pairs each source label with its reference GeoTIFF, reports instance IDs that are missing from or outside `1..num_craters`, and plots the reference beside a `tab20` instance mask. Missing-instance bounding boxes are outlined in red. The notebook is read-only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Rectangle
import numpy as np
import rasterio

REFERENCE_DIR = Path(
    "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/"
    "7_band_vis_uv/inst_seg/chips"
)
LABEL_DIR = Path(
    "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/"
    "7_band_vis_uv/inst_seg/labels"
)
REFERENCE_BAND = 1

SAMPLE_IDS = (
    "M1098029461CE_r2850_c450",
    "M1098043733CE_r750_c300",
    "M1098050897CE_r8250_c900",
    "M1098050897CE_r8550_c1050",
    "M1098058035CE_r600_c450",
)

In [ ]:
def resolve_one(root, sample_id, suffixes):
    matches = sorted(
        path
        for path in root.glob(f"{sample_id}*")
        if path.is_file() and path.suffix.lower() in suffixes
    )
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {suffixes} file for {sample_id} under "
            f"{root}; found {len(matches)}: {matches}"
        )
    return matches[0]


def load_sample(sample_id):
    label_path = resolve_one(LABEL_DIR, sample_id, {".npz"})
    reference_path = resolve_one(REFERENCE_DIR, sample_id, {".tif", ".tiff"})
    with np.load(label_path, allow_pickle=False) as archive:
        mask = np.asarray(archive["mask"]).copy()
        bboxes = np.asarray(archive["bboxes"]).copy()
        num_craters = int(np.asarray(archive["num_craters"]).item())
    with rasterio.open(reference_path) as dataset:
        reference = dataset.read(REFERENCE_BAND, masked=True).astype(np.float32)
    present = tuple(int(value) for value in np.unique(mask) if int(value) > 0)
    expected = set(range(1, num_craters + 1))
    missing = tuple(sorted(expected - set(present)))
    out_of_range = tuple(sorted(set(present) - expected))
    return {
        "sample_id": sample_id,
        "label_path": label_path,
        "reference_path": reference_path,
        "reference": reference,
        "mask": mask,
        "bboxes": bboxes,
        "num_craters": num_craters,
        "present": present,
        "missing": missing,
        "out_of_range": out_of_range,
    }


samples = [load_sample(sample_id) for sample_id in SAMPLE_IDS]
for sample in samples:
    print(sample["sample_id"])
    print(f"  reference:   {sample['reference_path']}")
    print(f"  label:       {sample['label_path']}")
    print(f"  num_craters: {sample['num_craters']}")
    print(f"  present:     {sample['present']}")
    print(f"  missing:     {sample['missing'] or 'none'}")
    print(f"  out of range:{sample['out_of_range'] or 'none'}")

In [ ]:
def display_limits(image):
    values = image.compressed() if np.ma.isMaskedArray(image) else image.ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0
    lower, upper = np.percentile(values, (2, 98))
    if lower == upper:
        upper = lower + 1.0
    return float(lower), float(upper)


def instance_coloring(num_craters):
    tab20 = plt.get_cmap("tab20")
    colors = [(0.0, 0.0, 0.0, 1.0)]
    colors.extend(tab20((instance_id - 1) % 20) for instance_id in range(1, num_craters + 1))
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(np.arange(-0.5, num_craters + 1.5), cmap.N)
    return cmap, norm


def add_boxes(axis, sample, *, missing_only=False):
    present = set(sample["present"])
    for instance_id, (x, y, width, height) in enumerate(sample["bboxes"], start=1):
        is_missing = instance_id not in present
        if missing_only and not is_missing:
            continue
        axis.add_patch(
            Rectangle(
                (x, y), width, height, fill=False,
                edgecolor="red" if is_missing else "white",
                linewidth=1.5 if is_missing else 0.6,
                linestyle="--" if is_missing else "-",
            )
        )
        if is_missing:
            axis.text(x, y, str(instance_id), color="red", fontsize=8, weight="bold")

In [ ]:
figure, axes = plt.subplots(len(samples), 2, figsize=(12, 4 * len(samples)), squeeze=False)
for row, sample in enumerate(samples):
    reference_axis, label_axis = axes[row]
    vmin, vmax = display_limits(sample["reference"])
    reference_axis.imshow(sample["reference"], cmap="gray", vmin=vmin, vmax=vmax)
    add_boxes(reference_axis, sample, missing_only=True)
    reference_axis.set_title(f"{sample['sample_id']} — reference band {REFERENCE_BAND}")

    maximum_id = max((sample["num_craters"], *sample["present"]))
    cmap, norm = instance_coloring(maximum_id)
    label_axis.imshow(sample["mask"], cmap=cmap, norm=norm, interpolation="nearest")
    add_boxes(label_axis, sample)
    missing = sample["missing"] or "none"
    outside = sample["out_of_range"] or "none"
    label_axis.set_title(
        f"tab20 instances — missing: {missing}; out of range: {outside}"
    )
    for axis in (reference_axis, label_axis):
        axis.set_axis_off()

figure.suptitle("Instance-label ID audit (red dashed boxes are missing from the mask)", fontsize=14)
figure.tight_layout()
plt.show()